# Quantum Commerce — Treinando o modelo de análise de sentimento no Google Colab

**O que você vai fazer aqui:** treinar, do zero, o modelo que classifica as avaliações da Quantum Commerce como **positiva**, **neutra** ou **negativa** — o mesmo modelo que o backend da loja usa no endpoint `POST /predict`.

**O que você vai aprender, passo a passo:**

1. Carregar um dataset real (Olist, ~100 mil avaliações em português)
2. Explorar os dados e transformar estrelas (1–5) em rótulos (negativo / neutro / positivo)
3. Transformar texto em números com TF-IDF
4. Treinar um *baseline* — o "chute honesto" que todo modelo precisa superar
5. Treinar o modelo principal: Regressão Logística
6. Comparar com uma rede neural simples (MLP)
7. Avaliar com F1-macro e matriz de confusão, e interpretar o resultado
8. Salvar o modelo treinado no Drive

**Como rodar:** no Colab, menu **Ambiente de execução → Executar tudo**. Não precisa de GPU — CPU comum basta, e o treino todo leva uns 5 minutos.

**De onde vêm os dados:** o arquivo `olist_order_reviews_dataset.csv` já está no seu Drive, na pasta `TRABALHO-FINAL/archive/`. O notebook monta o Drive e lê de lá — nada precisa ser baixado.

**Resultado esperado (para conferir no final):** Regressão Logística com acurácia ~0,78 e **F1-macro ~0,635**. Se chegar perto disso, o treino funcionou — são os mesmos números do treino oficial do projeto (`backend/ml/artifacts/metrics.json`), porque os hiperparâmetros aqui são os mesmos de lá.

## Passo 0 — O ambiente: o que já vem pronto no Colab

O Colab já traz Python com `pandas`, `scikit-learn` e `matplotlib` instalados. A célula abaixo só confere as versões — se um dia você rodar este notebook fora do Colab, é isso que precisa estar instalado.

In [ ]:
import sys
import sklearn
import pandas as pd
import matplotlib

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)

## Passo 1 — Montar o Google Drive e apontar para o arquivo

O Colab roda numa máquina do Google, que não enxerga o seu Drive sozinha. A célula abaixo pede a sua autorização e monta o Drive em `/content/drive` — é como "plugar" o seu Drive na máquina do Colab. Vai aparecer um link para autorizar e um campo para colar o código.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CAMINHO_CSV = '/content/drive/MyDrive/TRABALHO-FINAL/archive/olist_order_reviews_dataset.csv'
print("Arquivo encontrado!" if os.path.exists(CAMINHO_CSV) else "⚠️ Arquivo NÃO encontrado nesse caminho — leia a nota abaixo.")
print(CAMINHO_CSV)

> **Se o arquivo não foi encontrado:** confira no Drive se a pasta se chama mesmo `TRABALHO-FINAL` e se dentro dela há a pasta `archive` com o `olist_order_reviews_dataset.csv`. Se o nome for outro, ajuste o `CAMINHO_CSV` na célula acima. Alternativa: faça upload do CSV direto no Colab com `from google.colab import files; files.upload()` e aponte o caminho para `/content/olist_order_reviews_dataset.csv`.

## Passo 2 — Carregar o CSV e olhar por dentro

`read_csv` lê o arquivo para um DataFrame do pandas — uma tabela em memória. `shape` diz quantas linhas e colunas há; `head` mostra as primeiras linhas para conferir se as colunas são as esperadas.

In [ ]:
df = pd.read_csv(CAMINHO_CSV)
print("Formato (linhas, colunas):", df.shape)
print("\nColunas:", list(df.columns))
df.head(3)

## Passo 3 — Explorar: quantas avaliações por nota? E quantas têm texto?

Dois fatos definem o projeto inteiro:

- **As notas são muito desbalanceadas:** a nota 5 domina. Um modelo que sempre chutasse "positivo" já acertaria ~58% — por isso precisamos de uma métrica melhor que a acurácia (ver Passo 7).
- **Nem toda avaliação tem texto:** sem texto, não dá para treinar um modelo de texto. Essas linhas serão descartadas no próximo passo.

In [ ]:
print("Avaliações por nota (1 a 5):")
print(df['review_score'].value_counts().sort_index())
print("\nDatas:", df['review_creation_date'].min(), "→", df['review_creation_date'].max())

tem_titulo = df['review_comment_title'].notna()
tem_mensagem = df['review_comment_message'].notna()
print("\nCom título:", tem_titulo.sum(), "| Com mensagem:", tem_mensagem.sum())
print("Com título OU mensagem:", (tem_titulo | tem_mensagem).sum())
print("Sem texto nenhum:", (~(tem_titulo | tem_mensagem)).sum(), "— essas serão descartadas no Passo 4")

## Passo 4 — Construir o texto e o rótulo de cada avaliação

Duas transformações acontecem aqui:

- **Texto = título + mensagem.** O título sozinho salva 1.737 avaliações que não têm mensagem. Juntar os dois é o que o treino oficial faz.
- **Rótulo = nota convertida em sentimento:** notas 1–2 → `negativo`, nota 3 → `neutro`, notas 4–5 → `positivo`. É a mesma regra do Mercado Livre e a mesma do projeto.

Por que 3 classes e não 5 (uma por estrela)? Porque o que importa para a loja é o sentimento — e com 5 classes as fronteiras entre 4 e 5, ou entre 1 e 2, são ruído: o cliente que dá 4 ou 5 está satisfeito do mesmo jeito.

In [ ]:
def rotulo(nota):
    if nota <= 2:
        return "negativo"
    if nota == 3:
        return "neutro"
    return "positivo"

df['texto'] = (
    df['review_comment_title'].fillna('') + ' ' +
    df['review_comment_message'].fillna('')
).str.strip()
df['rotulo'] = df['review_score'].apply(rotulo)

dados = df[df['texto'] != ''][['texto', 'rotulo']].reset_index(drop=True)
print("Avaliações com texto:", len(dados), "de", len(df))
print("\nDistribuição dos rótulos:")
print(dados['rotulo'].value_counts())
print(dados['rotulo'].value_counts(normalize=True).round(3))
dados.head(3)

## Passo 5 — Separar treino e teste (o teste nunca é usado para treinar)

- **80% treino / 20% teste**, sorteados com `random_state=42` (a semente fixa deixa o experimento reproduzível: todo mundo que rodar chega nos mesmos números).
- **`stratify`**: o sorteio respeita a proporção de cada classe nos dois conjuntos — o teste tem os mesmos ~66% de positivos que o treino.

O teste fica guardado até o final. Ele é a nossa única medida honesta de "o modelo aprendeu, ou só decorou o treino?".

In [ ]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    dados['texto'], dados['rotulo'],
    test_size=0.20, random_state=42, stratify=dados['rotulo'],
)
print("Treino:", len(X_treino), "| Teste:", len(X_teste))
print("\nProporção no treino:")
print(y_treino.value_counts(normalize=True).round(3))
print("\nProporção no teste:")
print(y_teste.value_counts(normalize=True).round(3))

## Passo 6 — Transformar texto em números: TF-IDF

Modelos de machine learning não leem texto — leem números. O TF-IDF transforma cada avaliação num vetor:

- **Cada palavra (e cada par de palavras) vira uma coluna.** "não gostei" conta como uma feature só — por isso `ngram_range=(1, 2)`.
- **O peso de cada palavra** combina quantas vezes ela aparece na avaliação com o quão rara ela é no dataset inteiro. "produto" aparece em tudo e pesa pouco; "quebrado" é raro e pesa muito — é justamente o que distingue uma avaliação negativa.
- **`max_features=8000`**: só as 8.000 features mais informativas entram — o resto é ruído.
- **`min_df=2`**: palavra que aparece uma única vez no dataset inteiro é erro de digitação ou caso único — fora.
- **`sublinear_tf=True`**: atenua repetições ("muito muito muito bom" não vale 3× "muito bom").
- **`stop_words`**: palavras tão comuns ("de", "que", "o", "a") que não ajudam a distinguir sentimento — removidas antes de virar feature. A lista é a mesma do projeto (`backend/ml/stopwords.py`).

⚠️ **Detalhe que decide se o experimento é honesto:** `fit_transform` (aprender o vocabulário) é feito **só com o treino**. O teste passa por `transform` com o *mesmo* vocabulário. Se o vocabulário fosse aprendido com o teste junto, o modelo "veria" o teste antes da hora — é o chamado **vazamento de dados**.

In [ ]:
# Stopwords em português — as mesmas do projeto (backend/ml/stopwords.py).
STOPWORDS_PT = ['a', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'até', 'com', 'como', 'da', 'das', 'de', 'dela', 'delas', 'dele', 'deles', 'depois', 'do', 'dos', 'e', 'ela', 'elas', 'ele', 'eles', 'em', 'entre', 'era', 'eram', 'essa', 'essas', 'esse', 'esses', 'esta', 'estamos', 'estas', 'estava', 'estavam', 'este', 'estes', 'estou', 'eu', 'foi', 'foram', 'há', 'isso', 'isto', 'já', 'lhe', 'lhes', 'mais', 'mas', 'me', 'meu', 'meus', 'minha', 'minhas', 'muito', 'na', 'nas', 'nem', 'no', 'nos', 'nossa', 'nossas', 'nosso', 'nossos', 'num', 'numa', 'não', 'o', 'os', 'ou', 'para', 'pela', 'pelas', 'pelo', 'pelos', 'por', 'que', 'quem', 'se', 'seja', 'sejam', 'sem', 'seu', 'seus', 'sobre', 'sua', 'suas', 'só', 'também', 'te', 'tem', 'têm', 'todo', 'todos', 'um', 'uma', 'umas', 'uns', 'você', 'vocês', 'é', 'são', 'ser', 'está', 'estão', 'fui', 'vou', 'vai', 'vou', 'era', 'foi', 'quando', 'onde', 'qual', 'quais', 'quanto', 'quantos', 'porque', 'pois', 'então', 'ainda', 'mesmo', 'sempre', 'nunca', 'bem', 'mal', 'muito', 'pouco', 'cada', 'outro', 'outra', 'outros', 'outras', 'nada', 'tudo', 'algum', 'alguma', 'alguns', 'algumas', 'nenhum', 'nenhuma', 'esse', 'desse', 'dessa', 'disso', 'daqui', 'daí', 'lá', 'aqui', 'ali', 'agora', 'hoje', 'ontem', 'amanhã', 'antes', 'depois']

from sklearn.feature_extraction.text import TfidfVectorizer

vetorizador = TfidfVectorizer(
    ngram_range=(1, 2),        # palavras isoladas e pares ("não gostei")
    max_features=8000,         # só as 8.000 features mais informativas
    min_df=2,                  # ignora palavra que aparece 1 única vez
    sublinear_tf=True,         # atenua palavras repetidas muitas vezes
    stop_words=STOPWORDS_PT,   # remove "de", "que", "o", "a"...
    lowercase=True,            # "Ótimo" e "ótimo" viram a mesma palavra
)
X_treino_vec = vetorizador.fit_transform(X_treino)   # APRENDE o vocabulário SÓ com o treino
X_teste_vec = vetorizador.transform(X_teste)         # aplica o MESMO vocabulário ao teste
print("Vocabulário:", len(vetorizador.vocabulary_), "palavras/expressões")
print("Matriz de treino:", X_treino_vec.shape, "(linhas = avaliações, colunas = features)")

## Passo 7 — Baseline: o "chute honesto" que todo modelo precisa superar

Antes de qualquer modelo de verdade, treinamos um que **sempre prevê a classe mais comum** (positivo). Ele é o piso: se um modelo não ganhar dele, não aprendeu nada — só decorou a maioria.

A função `avaliar` abaixo será reutilizada para os três modelos. Ela mostra duas métricas:

- **Acurácia**: fração de acertos no total. Engana quando as classes são desbalanceadas (o baseline já faz 0,657 só chutando positivo).
- **F1-macro**: a média do F1 das três classes, cada classe valendo o mesmo. É a métrica principal do projeto — porque a classe neutra, com só 8,5% dos dados, vale tanto quanto a positiva.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

resultados = {}

def avaliar(nome, y_verdadeiro, y_previsto):
    acc = accuracy_score(y_verdadeiro, y_previsto)
    f1 = f1_score(y_verdadeiro, y_previsto, average='macro')
    resultados[nome] = {"accuracy": round(float(acc), 4), "f1_macro": round(float(f1), 4)}
    print(f"{nome:35s} acurácia={acc:.3f}  F1-macro={f1:.3f}")

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_treino_vec, y_treino)
pred_baseline = baseline.predict(X_teste_vec)
avaliar("Baseline (sempre positivo)", y_teste, pred_baseline)

## Passo 8 — O modelo principal: Regressão Logística

É o modelo do projeto. Para cada avaliação, ele calcula uma pontuação por classe a partir das palavras do TF-IDF e escolhe a maior. Dois parâmetros importam:

- **`C=4`**: controla a regularização. `C` maior = menos regularização = o modelo pode se ajustar mais aos dados de treino.
- **`class_weight='balanced'`**: compensa o desbalanceamento. A classe neutra, com só 8,5% do treino, passa a pesar mais no treino — sem isso, o modelo praticamente ignoraria os neutros.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(C=4, class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(X_treino_vec, y_treino)
pred_logreg = logreg.predict(X_teste_vec)
avaliar("Regressão Logística (TF-IDF)", y_teste, pred_logreg)

## Passo 9 — Comparação: uma rede neural simples (MLP)

Para cumprir o tópico 2 do enunciado (clássico × rede neural), comparamos com um MLP — uma rede com **uma camada oculta de 128 neurônios**, sobre o mesmo TF-IDF.

⚠️ **Por que a MLP treina numa subamostra de 12.000 linhas?** O MLP do scikit-learn não trabalha com matriz esparsa: ele densifica o TF-IDF. Densificar as ~34 mil linhas de treino × 8.000 features passaria de 2 GB de memória. A subamostra é estratificada (mantém a proporção das classes) e o teste continua sendo o mesmo para os três modelos — então a comparação segue válida.

In [ ]:
from sklearn.neural_network import MLPClassifier

# Subamostra estratificada de 12.000 linhas do treino (ver nota acima).
X_mlp, _, y_mlp, _ = train_test_split(
    X_treino_vec, y_treino, train_size=12000, random_state=42, stratify=y_treino,
)
mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=300, random_state=42)
mlp.fit(X_mlp, y_mlp)
pred_mlp = mlp.predict(X_teste_vec)
avaliar("MLP (1 camada, 128 neurônios)", y_teste, pred_mlp)

## Passo 10 — Avaliar e interpretar: onde o modelo erra?

O relatório por classe mostra precisão e recall de cada sentimento. E a matriz de confusão mostra **para onde vão os erros**: cada linha é o rótulo verdadeiro, cada coluna é o que o modelo previu. A diagonal é o acerto; fora dela, a confusão.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print(classification_report(y_teste, pred_logreg, target_names=['negativo', 'neutro', 'positivo']))

ordem = ['negativo', 'neutro', 'positivo']
cm = confusion_matrix(y_teste, pred_logreg, labels=ordem)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
ax.set_xticklabels(ordem); ax.set_yticklabels(ordem)
ax.set_xlabel('Previsto'); ax.set_ylabel('Verdadeiro')
ax.set_title('Matriz de confusão — Regressão Logística')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

## Passo 10 (interpretação) — O que os números dizem

- **O modelo principal funciona:** F1-macro ~0,635 contra 0,264 do baseline. Positivo é a classe mais fácil (F1 ~0,89) e negativo vem em seguida (~0,76).
- **A classe neutra é o gargalo:** F1 ~0,26. Olhando a matriz, os neutros verdadeiros se espalham: boa parte é prevista como positiva e outra como negativa. Isso não é (só) falha do modelo — a nota 3 é ambígua por natureza: "chegou, mas demorou", "ok, nada demais". Avaliações neutras são poucas (8,5%) e heterogêneas.
- **A MLP ganha em acurácia (~0,787) e perde em F1-macro (~0,600):** ela acerta mais no total porque acerta mais positivos, mas erra mais os neutros. É por isso que a métrica principal é o F1-macro, e não a acurácia — a decisão de negócio é proteger as classes minoritárias, não inflar o total.

## Passo 11 — Próximos experimentos (para brincar e aprender)

Cada item abaixo é uma mudança de uma linha — rode de novo e compare o F1-macro:

1. **Mudar o `C` da Regressão Logística** (1, 4, 10): `C` maior deixa o modelo mais flexível; `C` menor, mais conservador. O que acontece com o F1 da classe neutra?
2. **Mudar o `max_features` do TF-IDF** (4.000, 8.000, 20.000): mais features ajudam ou só trazem ruído?
3. **Tirar "não", "nunca" e "nem" da lista de stopwords:** hoje "não" é removido — então "não gostei" vira "gostei" antes de chegar no modelo. Tirar essas três palavras da lista e medir de novo é o experimento mais instrutivo deste notebook.
4. **Usar só a mensagem, sem o título** (ou vice-versa): quanto o título ajuda?
5. **Aumentar o `max_iter` da MLP ou a subamostra** (se a memória permitir): a rede melhora com mais dados?

## Passo 12 — Salvar o modelo treinado no Drive

O treino só vira produto quando o modelo sai do notebook. Aqui salvamos, no Drive, o vetorizador e a Regressão Logística (os mesmos arquivos que o backend carrega em `backend/ml/artifacts/`), mais um `metrics.json` com as métricas dos três modelos — para o `POST /predict` da loja usar exatamente este modelo.

In [ ]:
import json
import joblib

SAIDA = '/content/drive/MyDrive/TRABALHO-FINAL/modelo-sentimento/'
os.makedirs(SAIDA, exist_ok=True)

joblib.dump(vetorizador, SAIDA + 'tfidf_vectorizer.joblib')
joblib.dump(logreg, SAIDA + 'logreg.joblib')

metricas = {
    "modelos": resultados,
    "n_treino": int(len(X_treino)),
    "n_teste": int(len(X_teste)),
    "rotulos": {"1-2": "negativo", "3": "neutro", "4-5": "positivo"},
    "fonte": "olist_order_reviews_dataset.csv (Olist, 2016–2018)",
}
with open(SAIDA + 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

print("Salvo em:", SAIDA)
print(json.dumps(metricas, indent=2, ensure_ascii=False))

## Checklist — o que você deve conseguir explicar agora

- [ ] De onde vêm os dados, quantas avaliações têm texto e por que as sem texto foram descartadas
- [ ] Como estrelas viram rótulos de sentimento — e por que 3 classes, não 5
- [ ] O que o TF-IDF faz e por que o vocabulário é aprendido só com o treino
- [ ] Para que serve o baseline e por que o F1-macro é a métrica principal (e não a acurácia)
- [ ] O que `C` e `class_weight='balanced'` fazem na Regressão Logística
- [ ] Por que a MLP treina numa subamostra de 12.000 linhas
- [ ] Por que a classe neutra é o gargalo — e o que você tentaria para melhorá-la

**Onde isso entra no projeto:** o treino oficial vive em `backend/ml/train.py` (mesmos parâmetros, mesma fonte de dados) e os artefatos em `backend/ml/artifacts/` — o que você acabou de fazer aqui é o mesmo pipeline, só que interativo, célula por célula.